In [ ]:
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torchsummary import summary
import torch.optim as optim
import torch.nn as nn
import pandas as pd
import numpy as np
import torch
import timm

In [ ]:
# Custom dataset class for MNIST CSV files
class MNISTDataset(Dataset):
    def __init__(self, dataframe, is_train=True, transform=None):
        # Store training flag and transform
        self.is_train = is_train
        self.transform = transform

        # Convert dataframe to numpy
        data = dataframe.values

        # Separate labels and images if training
        if self.is_train:
            self.labels = data[:, 0].astype(np.int64)
            self.images = data[:, 1:].astype(np.float32).reshape(-1, 28, 28)
        else:
            self.labels = None
            self.images = data.astype(np.float32).reshape(-1, 28, 28)

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        image = self.images[idx]
    
        # Convert to tensor with a channel dimension
        image = torch.tensor(image).unsqueeze(0)  # shape: (1, 28, 28)
    
        if self.transform:
            image = self.transform(image)
    
        if self.is_train:
            label = torch.tensor(self.labels[idx])
            return image, label
        else:
            return image

In [ ]:
# Model using timm
class Model(nn.Module):
    def __init__(self, num_classes=10, pretrained=True):
        super(Model, self).__init__()

        # Load timm model
        self.model = timm.create_model(
            "regnety_004",
            pretrained=pretrained,
            in_chans=1,
            num_classes=num_classes,
            drop_rate=0.5,
        )

    def forward(self, x):
        return self.model(x)

In [ ]:
# Training function
def train_model(model, train_loader, criterion, optimizer, device, epochs=20):
    model.train()
    for epoch in range(epochs):
        total_loss = 0
        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            total_loss += loss.item()

        print(f"Epoch [{epoch+1}/{epochs}], Loss: {total_loss/len(train_loader):.4f}")

In [ ]:
# Prediction function
def predict(model, test_loader, device):
    model.eval()
    predictions = []
    with torch.no_grad():
        for images in test_loader:
            images = images.to(device)
            outputs = model(images)
            _, preds = torch.max(outputs, 1)
            predictions.extend(preds.cpu().numpy())
    return predictions

In [ ]:
# Main function
def main():
    # Load datasets
    train_df = pd.read_csv("/kaggle/input/digit-recognizer/train.csv")
    test_df = pd.read_csv("/kaggle/input/digit-recognizer/test.csv")

    # Define transforms
    train_transform = transforms.Compose([
        # Random affine transformation (rotation, shift, shear, zoom)
        transforms.RandomAffine(
            degrees=20,
            translate=(0.2, 0.2),
            shear=15,
            scale=(0.9, 1.1)
        ),
        transforms.ConvertImageDtype(torch.float32),
        transforms.Normalize((0.5,), (0.5,))
    ])
    
    test_transform = transforms.Compose([
        transforms.ConvertImageDtype(torch.float32),
        transforms.Normalize((0.5,), (0.5,))
    ])

    # Create dataset and dataloader
    train_dataset = MNISTDataset(train_df, is_train=True, transform=train_transform)
    test_dataset = MNISTDataset(test_df, is_train=False, transform=test_transform)

    # ✅ Use num_workers=0 to avoid Windows pickling issue
    train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=0)
    test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False, num_workers=0)

    # Set device
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    # Initialize timm ResNet14t model
    model = Model(num_classes=10, pretrained=True).to(device)

    # ✅ Print model summary
    summary(model, input_size=(1, 28, 28))

    # Loss and optimizer
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-3)

    # Train
    train_model(model, train_loader, criterion, optimizer, device)

    # Predict
    predictions = predict(model, test_loader, device)

    # Save submission file
    submission = pd.DataFrame({
        "ImageId": np.arange(1, len(predictions) + 1),
        "Label": predictions
    })
    submission.to_csv("/kaggle/working/submission.csv", index=False)
    print("Saved submission.csv")

In [ ]:
# Run main
if __name__ == "__main__":
    main()